<a href="https://colab.research.google.com/github/dtabuena/Resources/blob/main/Tools/mahlefy_vector_graphics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import re
import math
import pathlib
import warnings
import colorsys
import lxml.etree
import matplotlib as mpl

COLOR_PROPERTIES = ("fill", "stroke", "stop-color", "flood-color", "lighting-color", "color", "solid-color")
DEFAULT_BLACK_INHERITED_PROPERTIES = ("fill", "color")

# CSS named colors mapped to hex, e.g. "navy" -> "#000080"
NAMED_COLORS = {color_name.lower(): hex_value for color_name, hex_value in mpl.colors.CSS4_COLORS.items()}
NAMED_COLOR_ALTERNATION = "|".join(
    re.escape(color_name) for color_name in sorted(NAMED_COLORS, key=len, reverse=True)
)

COLOR_TOKEN_PATTERN = re.compile(
    r"#[0-9a-fA-F]{3,8}(?!\w)"
    r"|(?:rgba?|hsla?)\([^)]*\)"
    r"|(?<![#\w-])(?:" + NAMED_COLOR_ALTERNATION + r")(?![\w-])",
    re.IGNORECASE,
)
DECLARATION_PATTERN = re.compile(
    r"(?<![\w-])(" + "|".join(re.escape(property_name) for property_name in COLOR_PROPERTIES) + r")(\s*:\s*)([^;}]+)",
    re.IGNORECASE,
)
URL_PATTERN = re.compile(r"(url\([^)]*\))", re.IGNORECASE)
FUNCTION_PATTERN = re.compile(r"^(\w+)\((.*)\)$", re.DOTALL)
FONT_FACE_PATTERN = re.compile(r"(@font-face\s*\{[^}]*\})", re.IGNORECASE)
FONT_WEIGHT_PATTERN = re.compile(r"((?<![\w-])font-weight\s*:\s*)([^;}]+)", re.IGNORECASE)
FONT_SHORTHAND_PATTERN = re.compile(r"(?<![\w-])font\s*:\s*[^;}]+", re.IGNORECASE)


def classify_rgb(red_value, green_value, blue_value, tolerance):
    if max(red_value, green_value, blue_value) <= tolerance:
        return "black"
    if min(red_value, green_value, blue_value) >= 255 - tolerance:
        return "white"
    return None


def parse_channel(channel_token):
    channel_token = channel_token.strip()
    if channel_token.endswith("%"):
        channel_value = float(channel_token[:-1]) * 2.55
    else:
        channel_value = float(channel_token)
    return channel_value


def split_function_arguments(function_token):
    function_match = FUNCTION_PATTERN.match(function_token.strip())
    if function_match is None:
        raise ValueError(f"Could not parse color function '{function_token}'.")
    function_name = function_match.group(1)
    argument_text = function_match.group(2)
    if "/" in argument_text:
        channel_text, alpha_text = argument_text.split("/", 1)
        alpha_text = alpha_text.strip()
    else:
        channel_text = argument_text
        alpha_text = None
    argument_tokens = [argument_token for argument_token in re.split(r"[\s,]+", channel_text.strip()) if argument_token]
    if alpha_text is None and len(argument_tokens) == 4:
        alpha_text = argument_tokens.pop()
    if len(argument_tokens) != 3:
        raise ValueError(f"Expected 3 channels in color function '{function_token}'.")
    return function_name, argument_tokens, alpha_text


def parse_color_token(color_token):
    # returns (token_kind, channel_values on a 0-255 scale, alpha), or None for an invalid hex color
    lowered_token = color_token.lower()
    if lowered_token.startswith("#"):
        hex_digits = color_token[1:]
        digit_count = len(hex_digits)
        if digit_count in (3, 4):
            channel_width = 1
        elif digit_count in (6, 8):
            channel_width = 2
        else:
            return None
        channel_values = [
            float(int(hex_digits[channel_index * channel_width:(channel_index + 1) * channel_width] * (3 - channel_width), 16))
            for channel_index in range(3)
        ]
        alpha_digits = hex_digits[3 * channel_width:] * (3 - channel_width)
        parsed_color = ("hex", channel_values, alpha_digits)
    elif lowered_token.startswith("rgb"):
        function_name, argument_tokens, alpha_text = split_function_arguments(color_token)
        channel_values = [parse_channel(argument_token) for argument_token in argument_tokens]
        parsed_color = ("function", channel_values, alpha_text)
    elif lowered_token.startswith("hsl"):
        function_name, argument_tokens, alpha_text = split_function_arguments(color_token)
        hue_degrees = float(re.sub(r"deg$", "", argument_tokens[0].lower()))
        saturation_fraction = float(argument_tokens[1].rstrip("%")) / 100
        lightness_fraction = float(argument_tokens[2].rstrip("%")) / 100
        rgb_fractions = colorsys.hls_to_rgb((hue_degrees % 360) / 360, lightness_fraction, saturation_fraction)
        channel_values = [rgb_fraction * 255 for rgb_fraction in rgb_fractions]
        parsed_color = ("function", channel_values, alpha_text)
    elif lowered_token in NAMED_COLORS:
        parsed_color = parse_color_token(NAMED_COLORS[lowered_token])
    else:
        raise ValueError(f"Unrecognized color token '{color_token}'.")
    return parsed_color


def scale_lightness(channel_values, scale_brightness):
    clamped_fractions = [min(max(channel_value, 0.0), 255.0) / 255 for channel_value in channel_values]
    hue_fraction, lightness_fraction, saturation_fraction = colorsys.rgb_to_hls(*clamped_fractions)
    scaled_lightness = min(lightness_fraction * scale_brightness, 1.0)
    scaled_fractions = colorsys.hls_to_rgb(hue_fraction, scaled_lightness, saturation_fraction)
    scaled_channel_values = [scaled_fraction * 255 for scaled_fraction in scaled_fractions]
    return scaled_channel_values


def swap_color_token(color_token, tolerance, scale_brightness):
    parsed_color = parse_color_token(color_token)
    if parsed_color is None:
        warnings.warn(f"Invalid color '{color_token}' left unchanged.")
        return color_token
    token_kind, channel_values, alpha_text = parsed_color
    classification = classify_rgb(*channel_values, tolerance)
    if classification is None and scale_brightness == 1:
        return color_token

    if classification == "black":
        swapped_channel_values = [255.0, 255.0, 255.0]
    elif classification == "white":
        swapped_channel_values = [0.0, 0.0, 0.0]
    else:
        swapped_channel_values = channel_values
    if scale_brightness != 1:
        swapped_channel_values = scale_lightness(swapped_channel_values, scale_brightness)

    channel_integers = [int(round(min(max(channel_value, 0.0), 255.0))) for channel_value in swapped_channel_values]
    if token_kind == "hex":
        formatted_color = "#" + "".join(f"{channel_integer:02x}" for channel_integer in channel_integers) + alpha_text
    elif alpha_text is None:
        formatted_color = "rgb({}, {}, {})".format(*channel_integers)
    else:
        formatted_color = "rgba({}, {}, {}, {})".format(*channel_integers, alpha_text)
    return formatted_color


def swap_colors_in_value(value_text, tolerance, scale_brightness):
    # url(#id) references are protected so ids like #fff are not rewritten
    value_segments = URL_PATTERN.split(value_text)
    swapped_segments = []
    for segment_index, segment_text in enumerate(value_segments):
        if segment_index % 2 == 1:
            swapped_segments.append(segment_text)
        else:
            swapped_segment = COLOR_TOKEN_PATTERN.sub(
                lambda color_match: swap_color_token(color_match.group(0), tolerance, scale_brightness), segment_text
            )
            swapped_segments.append(swapped_segment)
    swapped_value = "".join(swapped_segments)
    return swapped_value


def swap_declarations(css_text, tolerance, scale_brightness):
    def replace_declaration(declaration_match):
        swapped_value = swap_colors_in_value(declaration_match.group(3), tolerance, scale_brightness)
        replaced_declaration = declaration_match.group(1) + declaration_match.group(2) + swapped_value
        return replaced_declaration

    swapped_css = DECLARATION_PATTERN.sub(replace_declaration, css_text)
    return swapped_css


def make_css_bold(css_text):
    # @font-face blocks describe the embedded font file itself, so their font-weight must stay accurate
    css_segments = FONT_FACE_PATTERN.split(css_text)
    bold_segments = []
    for segment_index, segment_text in enumerate(css_segments):
        if segment_index % 2 == 1:
            bold_segments.append(segment_text)
        else:
            weight_replaced_text = FONT_WEIGHT_PATTERN.sub(
                lambda weight_match: weight_match.group(1) + "bold", segment_text
            )
            # the font shorthand resets weight to normal when it omits one, so follow it with an explicit bold
            shorthand_replaced_text = FONT_SHORTHAND_PATTERN.sub(
                lambda shorthand_match: shorthand_match.group(0).rstrip() + "; font-weight: bold", weight_replaced_text
            )
            bold_segments.append(shorthand_replaced_text)
    bold_css = "".join(bold_segments)
    return bold_css


def apply_bold_to_tree(svg_root):
    for element in svg_root.iter():
        if not isinstance(element.tag, str):
            continue
        if element.get("font-weight") is not None:
            element.set("font-weight", "bold")
        style_value = element.get("style")
        if style_value is not None:
            element.set("style", make_css_bold(style_value))
        if lxml.etree.QName(element).localname == "style" and element.text is not None:
            element.text = make_css_bold(element.text)
    # font-weight inherits, so the root value covers text that has no weight of its own
    svg_root.set("font-weight", "bold")


def get_declared_value(element, property_name):
    style_pattern = re.compile(r"(?<![\w-])" + re.escape(property_name) + r"\s*:\s*([^;]+)")
    style_match = style_pattern.search(element.get("style", ""))
    if style_match is not None:
        declared_value = style_match.group(1).strip()
    else:
        declared_value = element.get(property_name)
    return declared_value


def process_element(element, tolerance, scale_brightness, inside_mask, inherited_originals):
    if not isinstance(element.tag, str):
        return
    local_name = lxml.etree.QName(element).localname

    current_originals = dict(inherited_originals)
    for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
        declared_value = get_declared_value(element, property_name)
        if declared_value is not None:
            current_originals[property_name] = declared_value

    if local_name == "mask":
        # mask luminance controls visibility, so its contents must not be swapped or scaled
        inside_mask = True
        for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
            if get_declared_value(element, property_name) is None:
                element.set(property_name, inherited_originals[property_name])

    if not inside_mask:
        for property_name in COLOR_PROPERTIES:
            attribute_value = element.get(property_name)
            if attribute_value is not None:
                element.set(property_name, swap_colors_in_value(attribute_value, tolerance, scale_brightness))
        style_value = element.get("style")
        if style_value is not None:
            element.set("style", swap_declarations(style_value, tolerance, scale_brightness))
        if local_name == "style" and element.text is not None:
            element.text = swap_declarations(element.text, tolerance, scale_brightness)
        swapped_default_color = swap_color_token("black", tolerance, scale_brightness)
        if local_name == "stop" and get_declared_value(element, "stop-color") is None:
            element.set("stop-color", swapped_default_color)
        if local_name in ("feFlood", "feDropShadow") and get_declared_value(element, "flood-color") is None:
            element.set("flood-color", swapped_default_color)

    for child_element in element:
        process_element(child_element, tolerance, scale_brightness, inside_mask, current_originals)


def swap_black_white_svg(input_path, output_path, tolerance=0, scale_brightness=1, bold_all=True):
    if not 0 <= tolerance < 128:
        raise ValueError("tolerance must be in [0, 128) so the black and white ranges do not overlap.")
    if not math.isfinite(scale_brightness) or scale_brightness <= 0:
        raise ValueError(f"scale_brightness must be a finite number greater than 0, got {scale_brightness!r}.")
    if not isinstance(bold_all, bool):
        raise TypeError(f"bold_all must be True or False, got {bold_all!r}.")
    input_path = pathlib.Path(input_path)
    output_path = pathlib.Path(output_path)
    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found: {input_path}")
    if input_path.suffix.lower() != ".svg":
        raise ValueError(f"Only .svg is supported, got '{input_path.suffix}' for {input_path}.")

    xml_parser = lxml.etree.XMLParser(remove_blank_text=False, strip_cdata=False, huge_tree=True)
    svg_tree = lxml.etree.parse(str(input_path), xml_parser)
    svg_root = svg_tree.getroot()
    if lxml.etree.QName(svg_root).localname != "svg":
        raise ValueError(f"Root element is not <svg> in {input_path}.")

    starting_originals = {"fill": "black", "color": "black"}
    process_element(svg_root, tolerance, scale_brightness, False, starting_originals)

    # unspecified fill and color default to black, so replace that default with its swapped color
    swapped_default_color = swap_color_token("black", tolerance, scale_brightness)
    for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
        if get_declared_value(svg_root, property_name) is None:
            svg_root.set(property_name, swapped_default_color)

    if bold_all:
        text_element_count = len(list(svg_root.iter("{*}text")))
        if text_element_count == 0:
            warnings.warn(
                f"{input_path.name}: bold_all is True but there are no <text> elements. "
                "Text converted to outlines cannot be bolded."
            )
        apply_bold_to_tree(svg_root)

    raster_image_count = len(list(svg_root.iter("{*}image")))
    if raster_image_count > 0:
        warnings.warn(f"{input_path.name}: {raster_image_count} embedded <image> element(s) were not changed.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    svg_tree.write(str(output_path), xml_declaration=True, encoding="utf-8")

In [19]:
import re
import math
import pathlib
import warnings
import colorsys
import lxml.etree
import matplotlib as mpl

COLOR_PROPERTIES = ("fill", "stroke", "stop-color", "flood-color", "lighting-color", "color", "solid-color")
DEFAULT_BLACK_INHERITED_PROPERTIES = ("fill", "color")
TEXT_ELEMENT_NAMES = ("text", "tspan", "textPath")

# CSS named colors mapped to hex, e.g. "navy" -> "#000080"
NAMED_COLORS = {color_name.lower(): hex_value for color_name, hex_value in mpl.colors.CSS4_COLORS.items()}
NAMED_COLOR_ALTERNATION = "|".join(
    re.escape(color_name) for color_name in sorted(NAMED_COLORS, key=len, reverse=True)
)

COLOR_TOKEN_PATTERN = re.compile(
    r"#[0-9a-fA-F]{3,8}(?!\w)"
    r"|(?:rgba?|hsla?)\([^)]*\)"
    r"|(?<![#\w-])(?:" + NAMED_COLOR_ALTERNATION + r")(?![\w-])",
    re.IGNORECASE,
)
DECLARATION_PATTERN = re.compile(
    r"(?<![\w-])(" + "|".join(re.escape(property_name) for property_name in COLOR_PROPERTIES) + r")(\s*:\s*)([^;}]+)",
    re.IGNORECASE,
)
URL_PATTERN = re.compile(r"(url\([^)]*\))", re.IGNORECASE)
FUNCTION_PATTERN = re.compile(r"^(\w+)\((.*)\)$", re.DOTALL)
FONT_FACE_PATTERN = re.compile(r"(@font-face\s*\{[^}]*\})", re.IGNORECASE)
FONT_WEIGHT_PATTERN = re.compile(r"((?<![\w-])font-weight\s*:\s*)([^;}]+)", re.IGNORECASE)
FONT_SHORTHAND_PATTERN = re.compile(r"(?<![\w-])font\s*:\s*[^;}]+", re.IGNORECASE)


def classify_rgb(red_value, green_value, blue_value, tolerance):
    if max(red_value, green_value, blue_value) <= tolerance:
        return "black"
    if min(red_value, green_value, blue_value) >= 255 - tolerance:
        return "white"
    return None


def parse_channel(channel_token):
    channel_token = channel_token.strip()
    if channel_token.endswith("%"):
        channel_value = float(channel_token[:-1]) * 2.55
    else:
        channel_value = float(channel_token)
    return channel_value


def split_function_arguments(function_token):
    function_match = FUNCTION_PATTERN.match(function_token.strip())
    if function_match is None:
        raise ValueError(f"Could not parse color function '{function_token}'.")
    function_name = function_match.group(1)
    argument_text = function_match.group(2)
    if "/" in argument_text:
        channel_text, alpha_text = argument_text.split("/", 1)
        alpha_text = alpha_text.strip()
    else:
        channel_text = argument_text
        alpha_text = None
    argument_tokens = [argument_token for argument_token in re.split(r"[\s,]+", channel_text.strip()) if argument_token]
    if alpha_text is None and len(argument_tokens) == 4:
        alpha_text = argument_tokens.pop()
    if len(argument_tokens) != 3:
        raise ValueError(f"Expected 3 channels in color function '{function_token}'.")
    return function_name, argument_tokens, alpha_text


def parse_color_token(color_token):
    # returns (token_kind, channel_values on a 0-255 scale, alpha), or None for an invalid hex color
    lowered_token = color_token.lower()
    if lowered_token.startswith("#"):
        hex_digits = color_token[1:]
        digit_count = len(hex_digits)
        if digit_count in (3, 4):
            channel_width = 1
        elif digit_count in (6, 8):
            channel_width = 2
        else:
            return None
        channel_values = [
            float(int(hex_digits[channel_index * channel_width:(channel_index + 1) * channel_width] * (3 - channel_width), 16))
            for channel_index in range(3)
        ]
        alpha_digits = hex_digits[3 * channel_width:] * (3 - channel_width)
        parsed_color = ("hex", channel_values, alpha_digits)
    elif lowered_token.startswith("rgb"):
        function_name, argument_tokens, alpha_text = split_function_arguments(color_token)
        channel_values = [parse_channel(argument_token) for argument_token in argument_tokens]
        parsed_color = ("function", channel_values, alpha_text)
    elif lowered_token.startswith("hsl"):
        function_name, argument_tokens, alpha_text = split_function_arguments(color_token)
        hue_degrees = float(re.sub(r"deg$", "", argument_tokens[0].lower()))
        saturation_fraction = float(argument_tokens[1].rstrip("%")) / 100
        lightness_fraction = float(argument_tokens[2].rstrip("%")) / 100
        rgb_fractions = colorsys.hls_to_rgb((hue_degrees % 360) / 360, lightness_fraction, saturation_fraction)
        channel_values = [rgb_fraction * 255 for rgb_fraction in rgb_fractions]
        parsed_color = ("function", channel_values, alpha_text)
    elif lowered_token in NAMED_COLORS:
        parsed_color = parse_color_token(NAMED_COLORS[lowered_token])
    else:
        raise ValueError(f"Unrecognized color token '{color_token}'.")
    return parsed_color


def scale_lightness(channel_values, scale_brightness):
    clamped_fractions = [min(max(channel_value, 0.0), 255.0) / 255 for channel_value in channel_values]
    hue_fraction, lightness_fraction, saturation_fraction = colorsys.rgb_to_hls(*clamped_fractions)
    scaled_lightness = min(lightness_fraction * scale_brightness, 1.0)
    scaled_fractions = colorsys.hls_to_rgb(hue_fraction, scaled_lightness, saturation_fraction)
    scaled_channel_values = [scaled_fraction * 255 for scaled_fraction in scaled_fractions]
    return scaled_channel_values


def swap_color_token(color_token, tolerance, scale_brightness):
    parsed_color = parse_color_token(color_token)
    if parsed_color is None:
        warnings.warn(f"Invalid color '{color_token}' left unchanged.")
        return color_token
    token_kind, channel_values, alpha_text = parsed_color
    classification = classify_rgb(*channel_values, tolerance)
    if classification is None and scale_brightness == 1:
        return color_token

    if classification == "black":
        swapped_channel_values = [255.0, 255.0, 255.0]
    elif classification == "white":
        swapped_channel_values = [0.0, 0.0, 0.0]
    else:
        swapped_channel_values = channel_values
    if scale_brightness != 1:
        swapped_channel_values = scale_lightness(swapped_channel_values, scale_brightness)

    channel_integers = [int(round(min(max(channel_value, 0.0), 255.0))) for channel_value in swapped_channel_values]
    if token_kind == "hex":
        formatted_color = "#" + "".join(f"{channel_integer:02x}" for channel_integer in channel_integers) + alpha_text
    elif alpha_text is None:
        formatted_color = "rgb({}, {}, {})".format(*channel_integers)
    else:
        formatted_color = "rgba({}, {}, {}, {})".format(*channel_integers, alpha_text)
    return formatted_color


def swap_colors_in_value(value_text, tolerance, scale_brightness):
    # url(#id) references are protected so ids like #fff are not rewritten
    value_segments = URL_PATTERN.split(value_text)
    swapped_segments = []
    for segment_index, segment_text in enumerate(value_segments):
        if segment_index % 2 == 1:
            swapped_segments.append(segment_text)
        else:
            swapped_segment = COLOR_TOKEN_PATTERN.sub(
                lambda color_match: swap_color_token(color_match.group(0), tolerance, scale_brightness), segment_text
            )
            swapped_segments.append(swapped_segment)
    swapped_value = "".join(swapped_segments)
    return swapped_value


def swap_declarations(css_text, tolerance, scale_brightness):
    def replace_declaration(declaration_match):
        swapped_value = swap_colors_in_value(declaration_match.group(3), tolerance, scale_brightness)
        replaced_declaration = declaration_match.group(1) + declaration_match.group(2) + swapped_value
        return replaced_declaration

    swapped_css = DECLARATION_PATTERN.sub(replace_declaration, css_text)
    return swapped_css


def make_css_bold(css_text):
    # @font-face blocks describe the embedded font file itself, so their font-weight must stay accurate
    css_segments = FONT_FACE_PATTERN.split(css_text)
    bold_segments = []
    for segment_index, segment_text in enumerate(css_segments):
        if segment_index % 2 == 1:
            bold_segments.append(segment_text)
        else:
            weight_replaced_text = FONT_WEIGHT_PATTERN.sub(
                lambda weight_match: weight_match.group(1) + "bold", segment_text
            )
            # the font shorthand resets weight to normal when it omits one, so follow it with an explicit bold
            shorthand_replaced_text = FONT_SHORTHAND_PATTERN.sub(
                lambda shorthand_match: shorthand_match.group(0).rstrip() + "; font-weight: bold", weight_replaced_text
            )
            bold_segments.append(shorthand_replaced_text)
    bold_css = "".join(bold_segments)
    return bold_css


def set_text_element_bold(text_element):
    # set bold both as an attribute and in the inline style, so it does not depend on inheritance
    text_element.set("font-weight", "bold")
    bold_style = make_css_bold(text_element.get("style", "")).strip().rstrip(";").strip()
    if FONT_WEIGHT_PATTERN.search(bold_style) is None:
        separator_text = "; " if bold_style else ""
        bold_style = bold_style + separator_text + "font-weight: bold"
    text_element.set("style", bold_style)


def apply_bold_to_tree(svg_root):
    bolded_text_count = 0
    for element in svg_root.iter():
        if not isinstance(element.tag, str):
            continue
        local_name = lxml.etree.QName(element).localname
        if local_name in TEXT_ELEMENT_NAMES:
            set_text_element_bold(element)
            bolded_text_count += 1
            continue
        if element.get("font-weight") is not None:
            element.set("font-weight", "bold")
        style_value = element.get("style")
        if style_value is not None:
            element.set("style", make_css_bold(style_value))
        if local_name == "style" and element.text is not None:
            element.text = make_css_bold(element.text)
    # the root value is a fallback for any text-like content not covered above
    svg_root.set("font-weight", "bold")
    return bolded_text_count


def get_declared_value(element, property_name):
    style_pattern = re.compile(r"(?<![\w-])" + re.escape(property_name) + r"\s*:\s*([^;]+)")
    style_match = style_pattern.search(element.get("style", ""))
    if style_match is not None:
        declared_value = style_match.group(1).strip()
    else:
        declared_value = element.get(property_name)
    return declared_value


def process_element(element, tolerance, scale_brightness, inside_mask, inherited_originals):
    if not isinstance(element.tag, str):
        return
    local_name = lxml.etree.QName(element).localname

    current_originals = dict(inherited_originals)
    for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
        declared_value = get_declared_value(element, property_name)
        if declared_value is not None:
            current_originals[property_name] = declared_value

    if local_name == "mask":
        # mask luminance controls visibility, so its contents must not be swapped or scaled
        inside_mask = True
        for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
            if get_declared_value(element, property_name) is None:
                element.set(property_name, inherited_originals[property_name])

    if not inside_mask:
        for property_name in COLOR_PROPERTIES:
            attribute_value = element.get(property_name)
            if attribute_value is not None:
                element.set(property_name, swap_colors_in_value(attribute_value, tolerance, scale_brightness))
        style_value = element.get("style")
        if style_value is not None:
            element.set("style", swap_declarations(style_value, tolerance, scale_brightness))
        if local_name == "style" and element.text is not None:
            element.text = swap_declarations(element.text, tolerance, scale_brightness)
        swapped_default_color = swap_color_token("black", tolerance, scale_brightness)
        if local_name == "stop" and get_declared_value(element, "stop-color") is None:
            element.set("stop-color", swapped_default_color)
        if local_name in ("feFlood", "feDropShadow") and get_declared_value(element, "flood-color") is None:
            element.set("flood-color", swapped_default_color)

    for child_element in element:
        process_element(child_element, tolerance, scale_brightness, inside_mask, current_originals)


def swap_black_white_svg(input_path, output_path, tolerance=0, scale_brightness=1, bold_all=True):
    if not 0 <= tolerance < 128:
        raise ValueError("tolerance must be in [0, 128) so the black and white ranges do not overlap.")
    if not math.isfinite(scale_brightness) or scale_brightness <= 0:
        raise ValueError(f"scale_brightness must be a finite number greater than 0, got {scale_brightness!r}.")
    if not isinstance(bold_all, bool):
        raise TypeError(f"bold_all must be True or False, got {bold_all!r}.")
    input_path = pathlib.Path(input_path)
    output_path = pathlib.Path(output_path)
    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found: {input_path}")
    if input_path.suffix.lower() != ".svg":
        raise ValueError(f"Only .svg is supported, got '{input_path.suffix}' for {input_path}.")

    xml_parser = lxml.etree.XMLParser(remove_blank_text=False, strip_cdata=False, huge_tree=True)
    svg_tree = lxml.etree.parse(str(input_path), xml_parser)
    svg_root = svg_tree.getroot()
    if lxml.etree.QName(svg_root).localname != "svg":
        raise ValueError(f"Root element is not <svg> in {input_path}.")

    starting_originals = {"fill": "black", "color": "black"}
    process_element(svg_root, tolerance, scale_brightness, False, starting_originals)

    # unspecified fill and color default to black, so replace that default with its swapped color
    swapped_default_color = swap_color_token("black", tolerance, scale_brightness)
    for property_name in DEFAULT_BLACK_INHERITED_PROPERTIES:
        if get_declared_value(svg_root, property_name) is None:
            svg_root.set(property_name, swapped_default_color)

    if bold_all:
        bolded_text_count = apply_bold_to_tree(svg_root)
        if bolded_text_count == 0:
            warnings.warn(
                f"{input_path.name}: bold_all is True but there are no text elements. "
                "Text converted to outlines cannot be bolded."
            )

    raster_image_count = len(list(svg_root.iter("{*}image")))
    if raster_image_count > 0:
        warnings.warn(f"{input_path.name}: {raster_image_count} embedded <image> element(s) were not changed.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    svg_tree.write(str(output_path), xml_declaration=True, encoding="utf-8")

In [20]:
import zipfile
import pathlib
import google.colab

output_directory = pathlib.Path("./SampleSVG_swapped")
results_zip_path = pathlib.Path("./SampleSVG_swapped.zip")

if not output_directory.is_dir():
    raise FileNotFoundError(f"Output folder not found: {output_directory.resolve()}")
swapped_paths = sorted(candidate_path for candidate_path in output_directory.rglob("*") if candidate_path.is_file())
if not swapped_paths:
    raise FileNotFoundError(f"No files found in {output_directory.resolve()}")

with zipfile.ZipFile(results_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as results_archive:
    for swapped_path in swapped_paths:
        archive_name = swapped_path.relative_to(output_directory).as_posix()
        results_archive.write(swapped_path, arcname=archive_name)

with zipfile.ZipFile(results_zip_path) as results_archive:
    archived_names = results_archive.namelist()
    corrupt_member_name = results_archive.testzip()
if corrupt_member_name is not None:
    raise zipfile.BadZipFile(f"Corrupt entry in {results_zip_path.name}: {corrupt_member_name}")
if len(archived_names) != len(swapped_paths):
    raise RuntimeError(f"Expected {len(swapped_paths)} files in {results_zip_path.name}, found {len(archived_names)}")

google.colab.files.download(str(results_zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>